In [ ]:
from pyspark.sql.functions import lit, col

In [ ]:
dbutils.widgets.text("catalog", "olist_project_dev")

dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("raw_olist_orders_reviews_table", "olist_orders_reviews")

dbutils.widgets.text("silver_schema", "olist_silver")
dbutils.widgets.text("orders_reviews_table", "orders_reviews_silver")

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_orders_reviews_table_name = dbutils.widgets.get("raw_olist_orders_reviews_table")

silver_schema = dbutils.widgets.get("silver_schema")
orders_reviews_table_name = dbutils.widgets.get("orders_reviews_table")

In [ ]:
raw_olist_orders_reviews_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_orders_reviews_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{raw_olist_orders_reviews_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{raw_olist_orders_reviews_table_name} (
            orderId STRING,
            reviewId STRING,
            reviewScore INT,
            reviewCommentTitle STRING,
            reviewCommentMessage STRING,
            reviewCreationDate TIMESTAMP,
            reviewAnswerTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
orders_reviews_silver_df = (
    raw_olist_orders_reviews_df
    .where((col("order_id").rlike("^[0-9a-fA-F]{32}$")))
    .select(
        col("order_id").alias("orderId"),
        col("review_id").alias("reviewId"),
        col("review_score").alias("reviewScore"),
        col("review_comment_title").alias("reviewCommentTitle"),
        col("review_comment_message").alias("reviewCommentMessage"),
        col("review_creation_date").alias("reviewCreationDate"),
        col("review_answer_timestamp").alias("reviewAnswerTimestamp")
    )
)

In [ ]:
orders_reviews_silver_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{silver_schema}.{orders_reviews_table_name}")